# 📊 Model Evaluation — 100 Random Samples

Runs both RetinaFace (detector) and U-Net (segmentor) on **100 randomly sampled images**
and reports per-model metrics + visualisations.

| Model | Dataset | Metric |
|-------|---------|--------|
| U-Net | CelebAMask-HQ test split | IoU, Dice, Pixel-Acc, Precision, Recall, F1 |
| RetinaFace | WIDER FACE val split | mAP@0.5, Recall@0.5 |

**Requirements:** `models/unet_final.pth`, `models/retinaface_final.pth`
**Runtime:** ~2–5 min on CPU (100 images, 256 px for seg, 640 px for det).

In [15]:
import sys, random, json, time
from pathlib import Path

REPO = Path('..').resolve()
sys.path.insert(0, str(REPO))

import numpy as np
import torch
import cv2
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # non-interactive backend

from tqdm.notebook import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

Device: cpu


In [16]:
# ── Helpers ────────────────────────────────────────────────────────────────

def sample_files(directory: Path, pattern='*.png', n=100):
    """Return n random files from a directory (shuffled, no replacement)."""
    files = sorted(Path(directory).glob(pattern))
    if not files:
        return []
    random.shuffle(files)
    return files[:n]

def ensure_dir(p):
    Path(p).mkdir(parents=True, exist_ok=True)
    return Path(p)

OUT_DIR = ensure_dir(REPO / 'runs' / 'evaluation' / 'notebook_100')
print(f'Output directory: {OUT_DIR}')

Output directory: E:\Face-Detection-Face-Segmentation\runs\evaluation\notebook_100


---

## 1. U-Net — Segmentation (100 random test images)

In [17]:
from src.segmentation.unet_model import UNet, UNetConfig

def load_unet(weights_path, device):
    model = UNet(UNetConfig(in_ch=3, out_ch=2, base_ch=64))
    state = torch.load(weights_path, map_location=device, weights_only=False)
    if isinstance(state, dict) and 'model_state' in state:
        state = state['model_state']
    model.load_state_dict(state, strict=False)
    return model.eval().to(device)

UNET_WEIGHTS = REPO / 'models' / 'unet_final.pth'
SEG_IMG_DIR  = REPO / 'data' / 'processed' / 'segmentation' / 'test' / 'images'
SEG_MASK_DIR = REPO / 'data' / 'processed' / 'segmentation' / 'test' / 'masks'
SEG_SIZE = 256

print(f'Weights  : {UNET_WEIGHTS}  exists={UNET_WEIGHTS.exists()}')
print(f'Image dir: {SEG_IMG_DIR}  exists={SEG_IMG_DIR.exists()}')
print(f'Mask dir : {SEG_MASK_DIR}  exists={SEG_MASK_DIR.exists()}')

Weights  : E:\Face-Detection-Face-Segmentation\models\unet_final.pth  exists=True
Image dir: E:\Face-Detection-Face-Segmentation\data\processed\segmentation\test\images  exists=True
Mask dir : E:\Face-Detection-Face-Segmentation\data\processed\segmentation\test\masks  exists=True


In [18]:
N_SEG = 100  # number of segmentation images

seg_files = sample_files(SEG_IMG_DIR, '*.png', N_SEG)
print(f'Found {len(seg_files)} images — sampling {min(N_SEG, len(seg_files))}')

Found 0 images — sampling 0


In [19]:
# ── Load model ─────────────────────────────────────────────────────────────
t0 = time.perf_counter()
unet = load_unet(UNET_WEIGHTS, DEVICE)
t_load = time.perf_counter() - t0
print(f'Model loaded in {t_load:.2f}s')

# ImageNet normalisation
NORM_MEAN = np.array([0.485, 0.456, 0.406])
NORM_STD  = np.array([0.229, 0.224, 0.225])

@torch.no_grad()
def segment_image(img_rgb: np.ndarray) -> np.ndarray:
    """Return binary face mask {0,1} at original resolution."""
    h0, w0 = img_rgb.shape[:2]
    img_resized = cv2.resize(img_rgb, (SEG_SIZE, SEG_SIZE))
    img_norm = ((img_resized / 255.0 - NORM_MEAN) / NORM_STD).astype(np.float32)
    tensor = torch.from_numpy(img_norm).permute(2, 0, 1).unsqueeze(0).to(DEVICE)
    pred = torch.argmax(unet(tensor), dim=1)[0].cpu().numpy().astype(np.uint8)
    return cv2.resize(pred, (w0, h0), interpolation=cv2.INTER_NEAREST)

Model loaded in 0.31s


In [22]:
# ── Per-image metrics + accumulation ──────────────────────────────────────
sum_iou = sum_dice = sum_pa = sum_p = sum_r = sum_f1 = 0.0
per_image = []
errors = []

t0 = time.perf_counter()
for img_path in tqdm(seg_files, desc='Segmentation'):
    mask_path = SEG_MASK_DIR / img_path.name
    if not mask_path.exists():
        errors.append(f'Missing mask: {mask_path.name}')
        continue

    img_bgr = cv2.imread(str(img_path))
    gt     = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if img_bgr is None or gt is None:
        errors.append(f'Read error: {img_path.name}')
        continue

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pred = segment_image(img_rgb)
    gt_bin = (cv2.resize(gt, (SEG_SIZE, SEG_SIZE), interpolation=cv2.INTER_NEAREST) > 127).astype(np.uint8)

    tp = int(((pred == 1) & (gt_bin == 1)).sum())
    fp = int(((pred == 1) & (gt_bin == 0)).sum())
    fn = int(((pred == 0) & (gt_bin == 1)).sum())
    tn = int(((pred == 0) & (gt_bin == 0)).sum())

    inter = tp
    union = tp + fp + fn
    iou  = inter / max(union, 1)
    dice = (2 * inter) / max(2 * inter + fp + fn, 1)
    pa   = (tp + tn) / max(tp + fp + fn + tn, 1)
    p    = tp / max(tp + fp, 1)
    r    = tp / max(tp + fn, 1)
    f1   = 2 * p * r / max(p + r, 1e-9)

    sum_iou  += iou
    sum_dice += dice
    sum_pa   += pa
    sum_p    += p
    sum_r    += r
    sum_f1   += f1
    per_image.append(dict(img=img_path.name, iou=iou, dice=dice, pa=pa, p=p, r=r, f1=f1))

seg_run_s = time.perf_counter() - t0
n = max(len(per_image), 1)

seg_metrics = dict(
    n_images=n,
    iou         = round(sum_iou  / n, 4),
    dice        = round(sum_dice / n, 4),
    pixel_acc   = round(sum_pa   / n, 4),
    precision   = round(sum_p    / n, 4),
    recall      = round(sum_r   / n, 4),
    f1          = round(sum_f1   / n, 4),
    runtime_s   = round(seg_run_s, 2),
)
print('\n=== Segmentation Metrics (100 random test images) ===')
for k, v in seg_metrics.items():
    print(f'  {k:<14}: {v}')
if errors:
    print(f'  errors        : {len(errors)} ({errors[:3]})')

ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html

In [ ]:
# ── Visualise best / worst / median samples ───────────────────────────────
sorted_by_iou = sorted(per_image, key=lambda x: x['iou'])
n_vis = min(9, n)

fig, axes = plt.subplots(3, 3, figsize=(14, 14))
axes = axes.flatten()

# Row 1: worst 3, Row 2: middle 3, Row 3: best 3 (guarded for small n)
worst_idx = list(range(min(3, n)))
mid_idx   = [max(0, min(n // 2 - 1, n - 1)),
            max(0, min(n // 2,     n - 1)),
            max(0, min(n // 2 + 1, n - 1))]
best_idx  = list(range(max(0, n - 3), n))
indices   = (worst_idx + mid_idx + best_idx)[:9]

for ax, idx in zip(axes, indices):
    item = sorted_by_iou[idx]
    img_path = [f for f in seg_files if f.name == item['img']][0]
    mask_path = SEG_MASK_DIR / item['img']
    img_bgr = cv2.imread(str(img_path))
    gt = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    pred = segment_image(img_rgb)
    gt_resized = cv2.resize(gt, (img_rgb.shape[1], img_rgb.shape[0]), interpolation=cv2.INTER_NEAREST)

    overlay = img_rgb.copy()
    overlay[pred == 1] = (0.6 * overlay[pred == 1] + 0.4 * np.array([255, 0, 0])).astype(np.uint8)

    ax.imshow(cv2.hConcat([
        cv2.hConcat([img_rgb, overlay]),
        cv2.hConcat([cv2.cvtColor(pred * 255, cv2.COLOR_GRAY2RGB),
                     cv2.cvtColor(gt_resized, cv2.COLOR_GRAY2RGB)])
    ]))
    ax.set_title(f"IoU={item['iou']:.3f}  {item['img']}", fontsize=8)
    ax.axis('off')

fig.suptitle('U-Net: Worst / Median / Best (Pred | GT)', fontsize=14, y=1.01)
plt.tight_layout()
vis_path = OUT_DIR / 'unet_100_samples.png'
plt.savefig(vis_path, dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Per-image IoU distribution histogram ─────────────────────────────────
ious = [x['iou'] for x in per_image]

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(ious, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
ax.axvline(np.mean(ious), color='red', linestyle='--', label=f'Mean={np.mean(ious):.4f}')
ax.axvline(np.median(ious), color='orange', linestyle=':', label=f'Median={np.median(ious):.4f}')
ax.set_xlabel('IoU'); ax.set_ylabel('Count')
ax.set_title('U-Net IoU distribution over 100 random test images')
ax.legend()
plt.tight_layout()
hist_path = OUT_DIR / 'unet_iou_histogram.png'
plt.savefig(hist_path, dpi=120)
plt.show()
print(f'Saved: {hist_path}')

---

## 2. RetinaFace — Detection (100 random val images)

In [ ]:
# ── Try detection dataset ──────────────────────────────────────────────────
DET_VAL_IMG = REPO / 'data' / 'processed' / 'detection' / 'val' / 'images'
DET_VAL_CSV = REPO / 'data' / 'processed' / 'detection' / 'val' / 'labels.csv'
DET_TEST_IMG = REPO / 'data' / 'processed' / 'detection' / 'test' / 'images'
DET_TEST_CSV = REPO / 'data' / 'processed' / 'detection' / 'test' / 'labels.csv'

det_img_dir = DET_VAL_IMG if DET_VAL_IMG.exists() else (DET_TEST_IMG if DET_TEST_IMG.exists() else None)
det_csv     = DET_VAL_CSV if DET_VAL_CSV.exists() else (DET_TEST_CSV if DET_TEST_CSV.exists() else None)

print(f'Detection val images : {DET_VAL_IMG}  exists={DET_VAL_IMG.exists()}')
print(f'Detection val csv    : {DET_VAL_CSV}  exists={DET_VAL_CSV.exists()}')
print(f'Using                : {det_img_dir}')

In [ ]:
from src.detection.retinaface import RetinaFace, load_retinaface_checkpoint
from src.detection.anchors import AnchorConfig, generate_anchors
from src.detection.eval import EvalEntry, load_gt, group_by_image, compute_map_recall

RF_WEIGHTS = REPO / 'models' / 'retinaface_final.pth'
IMG_SIZE = 640

print(f'RetinaFace weights: {RF_WEIGHTS}  exists={RF_WEIGHTS.exists()}')

In [ ]:
# ── Load RetinaFace ────────────────────────────────────────────────────────
t0 = time.perf_counter()
rf_model = RetinaFace()
rf_model = load_retinaface_checkpoint(str(RF_WEIGHTS))
rf_model.eval().to(DEVICE)
rf_load_s = time.perf_counter() - t0
print(f'RetinaFace loaded in {rf_load_s:.2f}s')

# Generate anchors (needed for decode)
anchor_cfg = AnchorConfig(image_size=IMG_SIZE, strides=(8, 16, 32))
anchors = generate_anchors(anchor_cfg)

NORM_MEAN_RF = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3, 1, 1)
NORM_STD_RF  = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3, 1, 1)

def preprocess_det(img_bgr: np.ndarray):
    h0, w0 = img_bgr.shape[:2]
    ratio = min(IMG_SIZE / h0, IMG_SIZE / w0)
    new_h, new_w = int(h0 * ratio), int(w0 * ratio)
    pad_w = IMG_SIZE - new_w
    pad_h = IMG_SIZE - new_h
    resized = cv2.resize(img_bgr, (new_w, new_h))
    padded  = cv2.copyMakeBorder(resized, 0, pad_h, 0, pad_w,
                               cv2.BORDER_CONSTANT, value=(0, 0, 0))
    rgb = cv2.cvtColor(padded, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    tensor = torch.from_numpy(rgb).permute(2, 0, 1)
    tensor = (tensor - torch.from_numpy(NORM_MEAN_RF)) / torch.from_numpy(NORM_STD_RF)
    return tensor, dict(ratio=ratio, pad_w=pad_w, pad_h=pad_h, h0=h0, w0=w0)

@torch.no_grad()
def detect_image(img_bgr: np.ndarray, conf_thr=0.02, nms_iou=0.5):
    """Return list of (xyxy_box, score)."""
    tensor, meta = preprocess_det(img_bgr)
    x = tensor.unsqueeze(0).to(DEVICE)
    out = rf_model(x)
    from src.detection.retinaface import flatten_predictions
    flat = flatten_predictions(out)

    cls_np = flat['cls_logits'][0].cpu().numpy()   # [N, 12]
    box_np = flat['box_deltas'][0].cpu().numpy()  # [N, 24]

    # sigmoid on class-1 channel (aggregate over 4 sub-channels per anchor)
    face_cls = cls_np[:, 1::4]                    # [N, 3]
    max_cls  = face_cls.max(axis=1)               # [N]
    scores   = 1.0 / (1.0 + np.exp(-max_cls))    # sigmoid

    # Decode boxes: xyxy from first anchor scale
    anc_np = anchors  # [N, 4]
    deltas = box_np[:, :4]                        # [N, 4]
    # Simple decode: scale * delta + anchor (approximate)
    scales = np.array([IMG_SIZE / 8, IMG_SIZE / 16, IMG_SIZE / 32])
    scales_rep = np.repeat(scales[:3], [len(anc_np) // 3] * 3)[:len(deltas)]
    decoded = np.zeros_like(deltas)
    decoded[:, 0] = anc_np[:len(deltas), 0] + deltas[:, 0] * scales_rep
    decoded[:, 1] = anc_np[:len(deltas), 1] + deltas[:, 1] * scales_rep
    decoded[:, 2] = anc_np[:len(deltas), 2] + deltas[:, 2] * scales_rep
    decoded[:, 3] = anc_np[:len(deltas), 3] + deltas[:, 3] * scales_rep
    decoded[:, [0, 2]] = np.clip(decoded[:, [0, 2]], 0, IMG_SIZE)
    decoded[:, [1, 3]] = np.clip(decoded[:, [1, 3]], 0, IMG_SIZE)
    xyxy = decoded

    # Filter by conf
    mask = scores >= conf_thr
    xyxy, sc = xyxy[mask], scores[mask]

    # Simple NMS
    keep = []
    order = np.argsort(sc)[::-1]
    while len(order) > 0 and len(keep) < 300:
        i = order[0]
        keep.append(i)
        if len(order) == 1:
            break
        ious = np.maximum(0, np.minimum(xyxy[order[1:]], [IMG_SIZE]*4) -
                              np.maximum(xyxy[order[1:]], [0]*4))
        ious = ious[:, 0] * ious[:, 3]
        order = order[1:][ious < nms_iou]

    xyxy, sc = xyxy[keep], sc[keep]
    return xyxy, sc

In [ ]:
# ── Run detection evaluation (or fallback to forward-pass only) ───────────

if det_img_dir and det_img_dir.exists():
    from src.detection.eval import load_gt

    gt_entries = load_gt(det_csv)
    gt_by_img  = group_by_image(gt_entries)
    all_img_ids = list(gt_by_img.keys())
    random.shuffle(all_img_ids)
    eval_img_ids = all_img_ids[:100]

    pred_entries = []
    per_det = []
    errors_det = []
    t0 = time.perf_counter()
    for img_id in tqdm(eval_img_ids, desc='RetinaFace detection'):
        img_path = det_img_dir / img_id
        img = cv2.imread(str(img_path))
        if img is None:
            errors_det.append(f'Read error: {img_id}')
            continue
        try:
            boxes, scores = detect_image(img, conf_thr=0.02, nms_iou=0.5)
        except Exception as e:
            errors_det.append(f'Detect error {img_id}: {e}')
            continue
        for box, score in zip(boxes, scores):
            pred_entries.append(EvalEntry(img_id, box.astype(np.float32), float(score)))
        per_det.append(dict(img=img_id, n_boxes=len(boxes), top_score=float(scores.max()) if len(scores) else 0.0))

    det_run_s = time.perf_counter() - t0

    # Compute mAP
    det_metrics = compute_map_recall(gt_entries, pred_entries, iou_threshold=0.5)
    det_metrics['n_images'] = len(eval_img_ids)
    det_metrics['runtime_s'] = round(det_run_s, 2)

    print('\n=== Detection Metrics (100 random val images) ===')
    for k, v in det_metrics.items():
        print(f'  {k:<16}: {v}')
    if errors_det:
        print(f'  errors           : {len(errors_det)}')
else:
    print('Detection dataset not found — running forward-pass diagnostic only.')
    print('Sample images will be generated synthetically.')
    det_metrics = None
    per_det = []
    det_run_s = 0.0
    eval_img_ids = []

In [ ]:
# ── Visualise detection on sample images ───────────────────────────────────
sample_imgs = []

if det_img_dir and det_img_dir.exists():
    # Pick 6 random images from the eval set
    sample_paths = [det_img_dir / i for i in eval_img_ids[:6] if (det_img_dir / i).exists()][:6]
else:
    # Synthesise 6 test images
    sample_paths = []
    for i in range(6):
        synth = (np.random.rand(640, 640, 3) * 255).astype(np.uint8)
        cv2.ellipse(synth, (320 + i * 10, 320), (100, 130), 0, 0, 360, (220, 200, 190), -1)
        cv2.ellipse(synth, (290 + i * 10, 290), (12, 8), 0, 0, 360, (30, 30, 30), -1)
        cv2.ellipse(synth, (350 + i * 10, 290), (12, 8), 0, 0, 360, (30, 30, 30), -1)
        p = OUT_DIR / f'synth_{i:02d}.png'
        cv2.imwrite(str(p), synth)
        sample_paths.append(p)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, path in zip(axes.flatten(), sample_paths):
    img = cv2.imread(str(path))
    if img is None:
        ax.set_title('Read error'); ax.axis('off'); continue
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h0, w0 = img.shape[:2]
    try:
        boxes, scores = detect_image(img, conf_thr=0.02)
    except Exception as e:
        ax.set_title(f'Error: {e}'); ax.imshow(img_rgb); ax.axis('off'); continue
    for box, sc in zip(boxes, scores):
        x1, y1, x2, y2 = box
        label = f'{sc:.2f}'
        cv2.rectangle(img_rgb, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
        cv2.putText(img_rgb, label, (int(x1), int(y1) - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    ax.imshow(img_rgb)
    ax.set_title(f'{path.name}  ({len(boxes)} dets)', fontsize=8)
    ax.axis('off')

fig.suptitle('RetinaFace — Sample detections (conf=0.02)', fontsize=14, y=1.01)
plt.tight_layout()
det_vis_path = OUT_DIR / 'retinaface_samples.png'
plt.savefig(det_vis_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {det_vis_path}')

---

## 3. Summary Dashboard

In [ ]:
# ── Save all results ───────────────────────────────────────────────────────
summary = {
    'timestamp'      : time.strftime('%Y-%m-%d %H:%M:%S'),
    'device'         : DEVICE,
    'n_samples'      : N_SEG,
    'segmentation'   : seg_metrics,
    'segmentation_per_image': per_image,
}

if det_metrics is not None:
    summary['detection'] = det_metrics
    summary['detection_per_image'] = per_det

summary_path = OUT_DIR / 'eval_100_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2, default=lambda x: float(x) if isinstance(x, np.floating) else x)
print(f'Summary saved: {summary_path}')

In [ ]:
# ── Pretty summary table ───────────────────────────────────────────────────
print('=' * 60)
print('  EVALUATION SUMMARY — 100 Random Samples')
print('=' * 60)

print('\n  U-Net Segmentation')
print('  ' + '-' * 44)
for k, v in seg_metrics.items():
    print(f'    {k:<16}: {v}')

if det_metrics is not None:
    print('\n  RetinaFace Detection')
    print('  ' + '-' * 44)
    for k, v in det_metrics.items():
        print(f'    {k:<16}: {v}')
    print('\n  Note: mAP / recall computed against WIDER FACE ground truth.')
    print('  mAP < 0.3 likely indicates the checkpoint was trained for < 20 epochs')
    print('  or uses random initialisation. Run full training on WIDER FACE to fix.')
else:
    print('\n  RetinaFace: detection dataset not found — forward-pass diagnostic only.')
    print('  Load WIDER FACE val split to compute mAP / recall.')

print('\n  Output files:')
for f in sorted(OUT_DIR.glob('*')):
    print(f'    {f.name}')
print('=' * 60)